# 6 · Saving and loading: snapshots

**The problem.** The address book has to be saved to disk, sent to a browser, and loaded back, possibly by a program
in another language. The usual approach is to write `to_dict` / `from_dict` per class and nest related records
inside each other. That breaks down on exactly the data from the last case studies:

- A shared address nested under two contacts is written twice and loads back as **two** addresses.
- A cycle (Byron mentors Ada, who is his mother) makes naive nesting recurse forever.
- A link filled by "a Person *or* an Institution" loses its kind unless you invent a type tag.

The framework serializes through one intermediate form, **plain data**: dicts, lists, strings, numbers, booleans and
`None`. JSON and YAML (case study 7) are thin text encodings of it. The shape is a *snapshot*: a flat table of objects
keyed by *symbols*, with links written as references to symbols. Nothing is nested, so nothing is duplicated and
cycles are harmless.

In [1]:
import json
import math

from mbse.Schemas.Framework import Plain, Proxies, Schemas, Validators
from mbse.Schemas.Framework.Errors import DecodeError
from toolkit import entries

store = Proxies.OfStore()

Contact = (
    Schemas.OfObject.Builder().ref()
    .properties(
        lambda prop: prop.name('name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('photo').of(lambda t: t.as_native(bytes)),
        lambda prop: prop.name('rating').of(lambda t: t.as_native(float)),
    )
    .create()
)
Address = Schemas.OfObject.Builder().ref().properties(lambda prop: prop.name('street').of(lambda t: t.as_native(str))).create()
ContactAddresses = (
    Schemas.OfRelation.Builder()
    .links('contact', 'address')
    .properties(lambda prop: prop.name('label').of(lambda t: t.as_native(str)))
    .create()
)
Contact = Schemas.OfObject.Builder(Contact).relations(lambda adj: adj.name('addresses').of(ContactAddresses).me('contact')).update()
Address = Schemas.OfObject.Builder(Address).relations(lambda adj: adj.name('residents').of(ContactAddresses).me('address')).update()
for name, schema in [('Contact', Contact), ('Address', Address), ('ContactAddresses', ContactAddresses)]:
    store.register(name, schema)
Builders = store

home = Builders.Address().street('7 Old Road').create()
alice = (
    Builders.Contact()
    .name('Alice')
    .photo(b'\x89PNG\r\n')
    .rating(math.inf)
    .addresses(lambda x: x.address(home).label('home'))
    .create()
)
reggie = Builders.Contact().name('Reginald').addresses(lambda x: x.address(home).label('home')).create()

## One object

`Plain.ToPlain(schema, value)` converts a value, given its schema:

In [2]:
Plain.ToPlain(store)(Contact, alice)

{'root': 's0',
 'objects': {'s0': {'name': 'Alice',
   'photo': 'iVBORw0K',
   'rating': 'Infinity',
   'addresses': [{'address': {'$ref': 's1', '$schema': 'Address'},
     'label': 'home'}]}}}

Read it carefully. Every part is a decision:

- **`root` and `objects`.** Objects live in a flat table under *symbols* (`s0`, `s1`, ...), assigned in the order
  objects are first referenced. `root` says which one you asked for.
- **No schema in the object.** `s0` doesn't say it's a `Contact`. You passed the schema in, and you'll pass it in
  again when loading. The data stays small, and the meaning comes from the schema, not from tags scattered through
  the data.
- **Native values in plain form.** `bytes` became base64 text and `inf` became the string `'Infinity'`, so the
  data fits in strict JSON. The conversion is done by the native type's schema, once, not by each text format.
- **Entries under the adjacency.** The entry lists the *other* links and the properties. The object's own link is
  implied by where the entry appears.
- **References carry the schema name.** `{'$ref': 's1', '$schema': 'Address'}` points at an object by symbol and
  says what it is. Links are untyped (case study 2), and a link may be filled by different kinds (case study 4), so
  the reference itself must say. This is why every linked object's schema must be registered under a name.

But `s1` isn't in `objects`. This snapshot covers *one* object, so its references dangle. That's fine for looking at
one record, but it can't be loaded on its own:

In [3]:
try:
    Plain.FromPlain(store)(Contact, Plain.ToPlain(store)(Contact, alice))
except DecodeError as error:
    print('DecodeError:', error)

DecodeError: $.objects.s0.addresses[0].address: unresolved reference 's1': the snapshot does not contain it


Loading refuses rather than inventing an empty address or dropping the entry. Either would silently change the
data. The error says *where* in the snapshot the problem is, as a path from the top (`$`); more on these errors
below.

## A whole graph

`Plain.ToPlain.Reachable` saves the root and everything reachable from it: the connected component from case study
4. Every reference now resolves:

In [4]:
graph = Plain.ToPlain(store).Reachable(Contact, alice)
print(json.dumps(graph, indent=2))

{
  "root": "s0",
  "objects": {
    "s0": {
      "name": "Alice",
      "photo": "iVBORw0K",
      "rating": "Infinity",
      "addresses": [
        {
          "address": {
            "$ref": "s1",
            "$schema": "Address"
          },
          "label": "home"
        }
      ]
    },
    "s1": {
      "street": "7 Old Road",
      "residents": [
        {
          "contact": {
            "$ref": "s0",
            "$schema": "Contact"
          },
          "label": "home"
        },
        {
          "contact": {
            "$ref": "s2",
            "$schema": "Contact"
          },
          "label": "home"
        }
      ]
    },
    "s2": {
      "name": "Reginald",
      "addresses": [
        {
          "address": {
            "$ref": "s1",
            "$schema": "Address"
          },
          "label": "home"
        }
      ]
    }
  }
}


The shared address appears **once** (`s1`), and both contacts refer to it. Each entry appears under *every* object
it links, from that object's point of view: under Alice as `address: s1`, and under the address as `contact: s0`.
That makes each object's record complete on its own. When loading, the duplicate views are recognized as one fact.

## Loading

`Plain.FromPlain(builders)` is constructed with the **builders** to build with, then called like `ToPlain`:

In [5]:
load = Plain.FromPlain(store)
alice2 = load.Reachable(Contact, graph)

print(alice2 is alice, alice2.name, alice2.photo, alice2.rating)
print([(e['address'].street, e['label']) for e in entries(alice2, 'addresses')])
[(e['contact'].name, e['label']) for e in entries(entries(alice2, 'addresses')[0]['address'], 'residents')]

False Alice b'\x89PNG\r\n' inf
[('7 Old Road', 'home')]


[('Alice', 'home'), ('Reginald', 'home')]

A new, separate graph came back: new objects, the same structure, each entry once, and the shared address shared.

### Why pass the builders in?

The loader doesn't create objects itself. It asks the registry you give it for a builder by schema name, and drives
that builder like any other code would. Today the registry is `Proxies.Builders`. When generated bindings exist,
you'll pass theirs and get typed classes back from the *same* snapshot. In tests you can pass a registry that
records or refuses. The serializer never needs to know which implementation is in use.

## Serializers check types

In case studies 1 and 5 the builder accepted a string where `bytes` belonged, and the validator reported it.
Serializers are the second checkpoint. They refuse to write data that doesn't match its schema, so a bad value
can't reach disk and come back as something else. Writing a wrong type is a mistake in *your program*, so it's an
ordinary `TypeError`:

In [6]:
sloppy = Builders.Contact().name('Bob').photo('not bytes').create()
try:
    Plain.ToPlain(store)(Contact, sloppy)
except TypeError as error:
    print('TypeError:', error)

# ## Loading errors: `DecodeError`
#
# Loading is just as strict. Types are never coerced: a JSON `1` doesn't become `1.0`, and `"1"` doesn't become `1`.
# But a wrong type in *loaded* data isn't a mistake in your program. It's a problem in the input, which your program
# must be ready for. So every problem found while loading raises one class, `Errors.DecodeError` (a `ValueError`),
# whichever layer finds it:

TypeError: expected bytes, got str


In [7]:
for bad in [{'rating': 1}, {'rating': 'fast'}, {'photo': 'not base64!'}, {'name': None}]:
    try:
        load(Contact, {'root': 's0', 'objects': {'s0': bad}})
    except DecodeError as error:
        print(f'{bad!s:28} -> {error}')

{'rating': 1}                -> $.objects.s0.rating: expected float, got int
{'rating': 'fast'}           -> $.objects.s0.rating: expected a float or one of ['-Infinity', 'Infinity', 'NaN'], got 'fast'
{'photo': 'not base64!'}     -> $.objects.s0.photo: invalid base64 text
{'name': None}               -> $.objects.s0.name: expected str, got NoneType


Each error has three parts you can use in code: `reason` (one line, no location), and either `path` (where in plain
data) or `line` and `column` (where in text, as case study 7 shows):

In [8]:
try:
    load(Contact, {'root': 's0', 'objects': {'s0': {'rating': 1}}})
except DecodeError as error:
    print(repr(error.path), '|', repr(error.reason), '|', error.line, error.column)

'$.objects.s0.rating' | 'expected float, got int' | None None


**Why one class?** A program loading a file wants to say "this file is bad, here's why and where" and carry on. With
errors from each underlying library (a `binascii.Error` from base64, a `JSONDecodeError`, a YAML parser's own
class), it would have to know every library the framework happens to use. With `DecodeError` it catches one class,
and the message reads the same in every language binding.

## Malformed snapshots are rejected, with a location

Snapshots come from outside your program, so their shape, references and values are all checked before anything is
built. If the check fails, nothing is built:

In [9]:
def mutated(change):
    snapshot = json.loads(json.dumps(graph))
    change(snapshot)
    return snapshot


for why, snapshot in [
    ('unknown property', mutated(lambda s: s['objects']['s0'].__setitem__('colour', 'red'))),
    ('missing object', mutated(lambda s: s['objects'].pop('s1'))),
    ('orphan object', mutated(lambda s: s['objects'].__setitem__('s9', {'street': '?'}))),
    ('wrong schema name', mutated(lambda s: s['objects']['s0']['addresses'][0]['address'].__setitem__('$schema', 'Contact'))),
]:
    try:
        load.Reachable(Contact, snapshot)
    except DecodeError as error:
        print(f'{why:18} -> {error}')

unknown property   -> $.objects.s0.colour: 'Contact' has no property or adjacency 'colour'
missing object     -> $.objects.s0.addresses[0].address: unresolved reference 's1': the snapshot does not contain it
orphan object      -> $.objects.s9: nothing references this object, so its schema is unknown
wrong schema name  -> $.objects.s2.addresses[0].address: 's1' is referenced as both 'Contact' and 'Address'


## Round trips

Saving what you loaded gives the same snapshot. Symbols may be *renumbered* when entries come back in a different
order, so compare structure, not symbol names, when order matters to you. (Whether snapshots should be fully
canonical is an open design question.)

In [10]:
again = Plain.ToPlain(store).Reachable(Contact, alice2)
print(again == graph)
print(Validators.Validate(store).Reachable(Contact, alice2))

True
[]


## Any schema, not just objects

`ToPlain` and `FromPlain` dispatch on the kind of schema you pass. They also have per-kind entry points
(`.OfNative`, `.OfObject`, `.Reachable`) when you want to be explicit:

In [11]:
Bytes = Schemas.OfNative.Data(bytes)
print(Plain.ToPlain(store)(Bytes, b'\x01\x02'), Plain.ToPlain(store).OfNative(Bytes, b'\x01\x02'), load.OfNative(Bytes, 'AQI='))

AQI= AQI= b'\x01\x02'


## What you learned

- A snapshot is `{'root': symbol, 'objects': {symbol: object}}`. Objects are flat, and links are
  `{'$ref': symbol, '$schema': name}`.
- The schema is passed in, not stored. References carry schema names because links are untyped.
- `ToPlain` saves one object (references dangle). `ToPlain.Reachable` saves a whole component.
- `FromPlain(builders)` builds with the builders you give it. It refuses dangling references, malformed shapes and
  wrong types with `DecodeError`, which carries a path, and it builds nothing when it refuses.
- Natives have plain forms: base64 for bytes, and `'NaN'`/`'Infinity'`/`'-Infinity'` for non-finite floats.

**Next:** case study 7 turns snapshots into JSON and YAML text, where the exactness above keeps paying off.